# 05.2 — Gradient accumulation

Sometimes the batch size you *want* (for stable gradients) is bigger than the batch your GPU can *hold*. Gradient accumulation resolves the conflict: split the big batch into GPU-sized micro-batches, run `backward()` on each **without clearing**, and step once at the end — the accumulated gradient equals the full-batch gradient. This is the *feature* side of the accumulation trap from [02.5 §2.3](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb), and this project's hardware-aware version (Critical Note #18).

This notebook covers:

* The memory problem, and why accumulation solves it exactly.
* The math: why summed micro-batch gradients equal the full-batch gradient.
* The correct loss weighting (mean vs sum).
* Hardware-aware sizing — `micro_batch_chunks` and the per-device table.

**Prerequisites:** [02.5 autograd](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb) (accumulation), [05.1 the loop](05.1_the_custom_training_loop.ipynb).

## Section 1 — What MATLAB does

The MATLAB pipeline (`cgg_procGradientAggregation`) accumulates gradients across sub-batches when the mini-batch won't fit, using a per-system table (`maxworkerMiniBatchSize`) to decide the chunk size:

```matlab
totalGrad = [];
for chunk = 1:NumChunks
    [~, grad] = dlfeval(@modelGradients, net, Xchunk, Tchunk);
    if isempty(totalGrad); totalGrad = grad;
    else; totalGrad = totalGrad + grad * chunkWeight; end    % accumulate
end
[net, ...] = adamupdate(net, totalGrad, ...);                 % ONE update
```

`maxworkerMiniBatchSize` varies by GPU (25 on a small card, 200 on a big one), so the same config runs everywhere by adapting the chunk size — the "hardware-aware" part. PyTorch gets accumulation for free from autograd's summing behavior; the port just has to chunk and weight correctly.

## Section 2 — The Python concepts you need

### 2.1 — The memory problem

A batch's memory is dominated by the *activations* stored for the backward pass — and that scales with batch size. Double the batch, double the activation memory; eventually the GPU OOMs. But small batches give noisier gradient estimates. Accumulation lets you keep the large *effective* batch while only ever holding a small one in memory.

### 2.2 — Why summed gradients equal the full-batch gradient

The key fact: `backward()` **adds** to `.grad` ([02.5 §2.3](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb)). So if you don't clear between micro-batches, gradients sum. And the gradient of a sum is the sum of gradients — so accumulating micro-batch gradients reconstructs the full-batch gradient *exactly*. Demonstrated:

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
X = torch.randn(8, 4)
y = torch.randn(8, 1)

def grad_of(model, idx):
    """Return the weight-gradient of model on examples `idx`, computed fresh."""
    model.zero_grad()
    loss = nn.functional.mse_loss(model(X[idx]), y[idx])
    loss.backward()
    return model[0].weight.grad.clone()

torch.manual_seed(1); full_model = nn.Linear(4, 1)
torch.manual_seed(1); accum_model = nn.Linear(4, 1)   # identical weights

# Full batch, one backward:
full_model.zero_grad()
nn.functional.mse_loss(full_model(X), y).backward()
full_grad = full_model.weight.grad.clone()

print("full-batch grad[0,:2]:", full_grad[0, :2].tolist())

In [ ]:
# Accumulate two micro-batches of 4 — WITHOUT zeroing between them, with correct weighting:
accum_model.zero_grad()
for idx in (slice(0, 4), slice(4, 8)):
    # weight = 4/8 = 0.5, because each micro-batch's MSE averages over 4, not 8:
    loss = nn.functional.mse_loss(accum_model(X[idx]), y[idx]) * (4 / 8)
    loss.backward()                        # ADDS to .grad — no zero_grad between!
accum_grad = accum_model.weight.grad.clone()

print("accumulated grad[0,:2]:", accum_grad[0, :2].tolist())
print("match full batch?", torch.allclose(full_grad, accum_grad, atol=1e-6))

Identical to the full-batch gradient — so a step taken from it is *mathematically the same update*, at a fraction of the memory. The training is unchanged; only the peak memory drops.

### 2.3 — The weighting subtlety (mean vs sum)

The `* (4/8)` above is the part people get wrong. `mse_loss` returns the **mean** over its batch, so a micro-batch of 4 averages over 4 while the full batch averages over 8. Without reweighting, each micro-batch's gradient would be too large by the ratio of sizes. The fix: weight each micro-batch's loss by its **fraction of the full batch** (`chunk_size / total`), so the weighted means sum to the full mean:

In [ ]:
# Wrong (no weighting) vs right (fractional weight) — the wrong one is 2× too big:
accum_model.zero_grad()
for idx in (slice(0, 4), slice(4, 8)):
    nn.functional.mse_loss(accum_model(X[idx]), y[idx]).backward()    # NO weight
wrong = accum_model.weight.grad[0, 0].item()
print(f"unweighted accumulation: {wrong:.4f}")
print(f"full-batch (target):     {full_grad[0, 0].item():.4f}")
print(f"ratio: {wrong / full_grad[0,0].item():.1f}× too big — exactly the 8/4 mismatch")

### 2.4 — Hardware-aware chunking

The project's `micro_batch_chunks` yields `(start, end, weight)` triples that partition a batch — each chunk ≤ the device's max size, each carrying its fractional weight. It's the §2.3 arithmetic, automated:

In [ ]:
from neural_data_decoding.training.accumulation import micro_batch_chunks

print("no accumulation (max_size=None):", list(micro_batch_chunks(8, None)))
print("even split (8 into 4s):         ", list(micro_batch_chunks(8, 4)))
print("uneven (7 into 3s):             ", [(s, e, round(w, 3)) for s, e, w in micro_batch_chunks(7, 3)])

The weights sum to 1.0 in every case (even the uneven split: 0.429 + 0.429 + 0.143 = 1.0), so the accumulated gradient always reconstructs the full-batch gradient. And `max_size=None` yields a single full-batch chunk — the no-accumulation fast path is the *same code*, just one chunk. The per-device max comes from `get_accumulation_size_for_current_system`, which reads the config's `accumulation_information` table and looks up the detected GPU (or `"CPU"`) — the MATLAB `maxworkerMiniBatchSize` mechanism.

In [ ]:
from neural_data_decoding.training.accumulation import get_accumulation_size_for_current_system

# The base.yaml table maps device names → max micro-batch:
table = {"CPU": 100, "NVIDIA RTX A6000": 20, "NVIDIA TITAN Xp": 20}
print("resolved max micro-batch for THIS machine:",
      get_accumulation_size_for_current_system(table))

## Section 3 — The `neural_data_decoding` implementation

`micro_batch_chunks` — the partition-and-weight generator, docstring first:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/accumulation.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("def micro_batch_chunks"))
print(f"{i + 1:4} | {src[i]}")            # the signature
print("     |   ... (docstring) ...")
body = next(j for j in range(i, len(src)) if src[j].strip().startswith("if n_total"))
for k in range(body, min(body + 12, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* The single-chunk fast path (`max_size is None or max_size >= n_total`) returns `[(0, n_total, 1.0)]` — accumulation-off and accumulation-on share one code path, so there's no separate branch to keep in sync.
* Each weight is `chunk_len / n_total` — §2.3's fractional weighting, so the accumulated gradient is exact regardless of even/uneven splits.
* It's a *generator* ([01.2](../01_python_for_matlab_users/01.2_control_flow.ipynb)) — the training loop iterates it, running one forward/backward per chunk without ever materializing the full batch's activations.

## Section 4 — Hands-on exercises

### Exercise 4.1 — weights sum to one

Confirm that `micro_batch_chunks` weights always sum to 1.0, for a spread of sizes and chunk limits.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
for n, m in [(10, 3), (16, 4), (7, 2), (100, 7), (5, None)]:
    total_w = sum(w for _, _, w in micro_batch_chunks(n, m))
    print(f"  n={n:3}, max={str(m):4} → weights sum to {total_w:.6f}")

### Exercise 4.2 — accumulate a real gradient

Split an 8-example batch into 4 micro-batches of 2 using `micro_batch_chunks`, accumulate the weighted gradient, and confirm it matches the full-batch gradient.

In [ ]:
# Reveal:
torch.manual_seed(1); model = nn.Linear(4, 1)
model.zero_grad()
for start, end, weight in micro_batch_chunks(8, 2):
    loss = nn.functional.mse_loss(model(X[start:end]), y[start:end]) * weight
    loss.backward()
accumulated = model.weight.grad.clone()
print("matches full-batch grad?", torch.allclose(accumulated, full_grad, atol=1e-6))

## Section 5 — Common errors

### Accumulated gradient is N× too large

Missing the fractional loss weight (§2.3) — each micro-batch's mean isn't rescaled to its share of the full batch. Weight by `chunk_size / total`, or use `micro_batch_chunks`'s weights.

### `zero_grad()` inside the micro-batch loop

That defeats accumulation — you clear the previous chunk's gradient before it can add. `zero_grad()` goes *before* the chunk loop, `step()` *after*; nothing clears in between.

### Still OOM despite accumulation

The chunk size is still too big, or something outside the loop (the model, a cached tensor) dominates memory. Lower `max_size`; check you're not retaining the graph across chunks (each chunk's `backward()` frees its own).

### Accumulation changes results vs full batch

It shouldn't (§2.2) — if it does, either the weighting is wrong (§2.3) or a batch-norm layer is involved (its statistics DO depend on the actual batch seen, so micro-batches shift them — [05.7](05.7_batch_norm_state_synchronization.ipynb)).

### The single-chunk path behaves differently

It shouldn't — `max_size=None` is one chunk with weight 1.0, identical to no accumulation. If it differs, you have a separate no-accum code path that's drifted; the project uses one path for both.

## Section 6 — Further reading

- [Gradient accumulation explained](https://kozodoi.me/blog/20210219/gradient-accumulation) — the memory/effective-batch tradeoff with diagrams.
- [`src/neural_data_decoding/training/accumulation.py`](../../src/neural_data_decoding/training/accumulation.py) — `micro_batch_chunks` + the device-table resolver.
- [`tests/unit/test_accumulation.py`](../../tests/unit/test_accumulation.py) — the parity tests, including the gradient-equivalence check.

Next up: **[05.3 gradient clipping](05.3_gradient_clipping.ipynb)** — keeping gradients from exploding, and the Global vs SubNetwork clip.